In [3]:
import cv2
import os
import time

# --- CONFIGURATION DES CHEMINS ---
BASE_PATH = r"C:\Users\PC\Documents\Active_Learning\videos_B"

def enregistrer_geste(session, geste, duree_secondes):
    cap = cv2.VideoCapture(0)
    save_dir = os.path.join(BASE_PATH, session, geste)
    os.makedirs(save_dir, exist_ok=True)
    
    print(f"\n🎥 Session: {session.upper()} | Geste: {geste}")
    print(f"Préparez-vous... Enregistrement de {duree_secondes}s dans 10 secondes.")
    time.sleep(10)
    
    start_time = time.time()
    count = 0
    
    while (time.time() - start_time) < duree_secondes:
        ret, frame = cap.read()
        if not ret: break
        
        # Redimensionnement immédiat à 224x224 (Format ResNet-18) 
        frame_resized = cv2.resize(frame, (224, 224))
        
        # Sauvegarde d'environ 5 images par seconde pour éviter les doublons inutiles
        if count % 6 == 0: 
            img_name = f"img_{int(time.time()*1000)}.jpg"
            cv2.imwrite(os.path.join(save_dir, img_name), frame_resized)
        
        # Affichage
        cv2.putText(frame, f"REC {geste} - {int(time.time()-start_time)}s", (10, 30), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
        cv2.imshow("Capture Domaine B", frame)
        count += 1
        if cv2.waitKey(1) & 0xFF == ord('q'): break
            
    cap.release()
    cv2.destroyAllWindows()
    print(f"✅ Terminé pour {geste}")

# --- EXÉCUTION DU PLAN DE TOURNAGE ---

# --- EXÉCUTION DU PLAN DE TOURNAGE CORRIGÉ ---

# 1. Pour le TEST B (6 vidéos de 15s au total)
# On fait 2 passages pour avoir 2 vidéos par geste comme demandé
for i in range(2): 
    for g in ["0_Pierre", "1_Feuille", "2_Ciseaux"]:
        print(f"\nVague de capture {i+1} pour le TEST (Changez de fond/lumière !)")
        enregistrer_geste("test", g, 15) 

# 2. Pour le POOL (12 vidéos de 10s au total)
# On change range(2) en range(4) pour avoir 4 vidéos par geste
for i in range(4): 
    for g in ["0_Pierre", "1_Feuille", "2_Ciseaux"]:
        print(f"\nVague de capture {i+1} pour le Pool")
        enregistrer_geste("pool", g, 10)


Vague de capture 1 pour le TEST (Changez de fond/lumière !)

🎥 Session: TEST | Geste: 0_Pierre
Préparez-vous... Enregistrement de 15s dans 10 secondes.
✅ Terminé pour 0_Pierre

Vague de capture 1 pour le TEST (Changez de fond/lumière !)

🎥 Session: TEST | Geste: 1_Feuille
Préparez-vous... Enregistrement de 15s dans 10 secondes.
✅ Terminé pour 1_Feuille

Vague de capture 1 pour le TEST (Changez de fond/lumière !)

🎥 Session: TEST | Geste: 2_Ciseaux
Préparez-vous... Enregistrement de 15s dans 10 secondes.
✅ Terminé pour 2_Ciseaux

Vague de capture 2 pour le TEST (Changez de fond/lumière !)

🎥 Session: TEST | Geste: 0_Pierre
Préparez-vous... Enregistrement de 15s dans 10 secondes.
✅ Terminé pour 0_Pierre

Vague de capture 2 pour le TEST (Changez de fond/lumière !)

🎥 Session: TEST | Geste: 1_Feuille
Préparez-vous... Enregistrement de 15s dans 10 secondes.
✅ Terminé pour 1_Feuille

Vague de capture 2 pour le TEST (Changez de fond/lumière !)

🎥 Session: TEST | Geste: 2_Ciseaux
Préparez-vou

In [1]:
pip install opencv-python

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\PC\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [1]:
import sys
!{sys.executable} -m pip install torchvision torch


  Using cached torchvision-0.25.0-cp310-cp310-win_amd64.whl (3.7 MB)


You should consider upgrading via the 'c:\Users\PC\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [4]:
# =================================================================
# ÉTAPE 3 : PRÉPARATION ET CHARGEMENT DES DONNÉES (A et B)
# =================================================================

import os
import cv2
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
from pathlib import Path
import torchvision.transforms as transforms

# --- 1. CONFIGURATION DES CHEMINS ---
BASE_DIR = Path(r"C:\Users\PC\Documents\Active_Learning")
PATH_DOMAIN_A = BASE_DIR / "archive"
PATH_DOMAIN_B = BASE_DIR / "videos_B" 

# On sépare les noms de dossiers pour gérer la différence de langue/nommage
CLASSES_A = ["rock", "paper", "scissors"]
LABEL_MAP_A = {"rock": 0, "paper": 1, "scissors": 2}

CLASSES_B = ["0_Pierre", "1_Feuille", "2_Ciseaux"]
LABEL_MAP_B = {"0_Pierre": 0, "1_Feuille": 1, "2_Ciseaux": 2}

IMG_SIZE = (224, 224) 

# --- 2. FONCTION DE CHARGEMENT GÉNÉRIQUE ---
def load_images_local(root_path, subset, current_classes, current_label_map, resize=IMG_SIZE):
    X, y = [], []
    subset_path = Path(root_path) / subset
    
    print(f"Chargement de {subset} dans {root_path}...")
    if not subset_path.exists():
        print(f"⚠️ Dossier inexistant : {subset_path}")
        return np.array(X), np.array(y)

    for cls_name in current_classes:
        cls_folder = subset_path / cls_name
        if not cls_folder.exists(): 
            print(f"   ⚠️ Dossier {cls_name} non trouvé dans {subset}")
            continue

        for img_name in os.listdir(cls_folder):
            img_path = cls_folder / img_name
            img = cv2.imread(str(img_path))
            if img is not None:
                # ResNet attend du RGB (OpenCV lit en BGR)
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, resize)
                X.append(img)
                y.append(current_label_map[cls_name])

    return np.array(X), np.array(y)

# --- 3. CHARGEMENT RÉEL DES DONNÉES ---
X_train_A, y_train_A = load_images_local(PATH_DOMAIN_A, "train", CLASSES_A, LABEL_MAP_A)
X_test_A, y_test_A   = load_images_local(PATH_DOMAIN_A, "test", CLASSES_A, LABEL_MAP_A)

X_pool_B, y_pool_B   = load_images_local(PATH_DOMAIN_B, "pool", CLASSES_B, LABEL_MAP_B)
X_test_B, y_test_B   = load_images_local(PATH_DOMAIN_B, "test", CLASSES_B, LABEL_MAP_B)

# --- 4. CONVERSION EN TENSEURS PYTORCH ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def to_tensor(X, y):
    if len(X) == 0: return None
    
    # Transformation en tenseur (Batch, Channel, Height, Width) et normalisation 0-1
    tensor_x = torch.Tensor(X).permute(0, 3, 1, 2) / 255.0
    
    # Normalisation spécifique à ResNet (Moyenne et écart-type d'ImageNet)
    # Cela aide le modèle pré-entraîné à reconnaître les formes
    norm = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    tensor_x = torch.stack([norm(t) for t in tensor_x])
    
    tensor_y = torch.LongTensor(y)
    return TensorDataset(tensor_x.to(device), tensor_y.to(device))

# Création des Datasets
train_A_ds = to_tensor(X_train_A, y_train_A)
test_A_ds  = to_tensor(X_test_A, y_test_A)
pool_B_ds  = to_tensor(X_pool_B, y_pool_B)
test_B_ds  = to_tensor(X_test_B, y_test_B)

print(f"\n📊 RÉSUMÉ DES DONNÉES (Conforme Tableau 2 du Proposal) :")
print(f"Domaine A (Kaggle) : Train={len(X_train_A)}, Test={len(X_test_A)}")
print(f"Domaine B (Webcam) : Pool={len(X_pool_B)}, Test={len(X_test_B)}")

Chargement de train dans C:\Users\PC\Documents\Active_Learning\archive...
Chargement de test dans C:\Users\PC\Documents\Active_Learning\archive...
Chargement de pool dans C:\Users\PC\Documents\Active_Learning\videos_B...
Chargement de test dans C:\Users\PC\Documents\Active_Learning\videos_B...

📊 RÉSUMÉ DES DONNÉES (Conforme Tableau 2 du Proposal) :
Domaine A (Kaggle) : Train=1020, Test=540
Domaine B (Webcam) : Pool=741, Test=619


In [5]:
# =================================================================
# BLOC 2 : MODÈLE, BASELINE ET PARAMÈTRES
# =================================================================

import torch
import torch.nn as nn
import torchvision.models as models
import time

# --- 1. DÉFINITION DU MODÈLE (ResNet-18) ---
class GestureClassifier(nn.Module):
    def __init__(self):
        super(GestureClassifier, self).__init__()
        # Chargement des poids ImageNet (transfert learning)
        self.resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        
        # Modification de la dernière couche pour 3 classes (Pierre, Feuille, Ciseaux)
        num_ftrs = self.resnet.fc.in_features
        self.resnet.fc = nn.Linear(num_ftrs, 3) 

    def forward(self, x):
        return self.resnet(x)

# --- 2. FONCTIONS D'ENTRAÎNEMENT ET ÉVALUATION ---

def train_model(model, train_loader, criterion, optimizer, epochs=5, device='cpu'):
    model.to(device)
    model.train() # Mode entraînement
    
    print(f"🚀 Début de l'entraînement ({epochs} époques)...")
    start_time = time.time()
    
    for epoch in range(epochs):
        running_loss = 0.0
        correct = 0
        total = 0
        
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
        epoch_acc = 100. * correct / total
        print(f"   Époque {epoch+1}/{epochs} | Loss: {running_loss/total:.4f} | Acc Train: {epoch_acc:.2f}%")
        
    print(f"✅ Entraînement terminé en {time.time() - start_time:.1f}s")
    return model

def evaluate_model(model, loader, name="Test"):
    model.eval()
    correct = 0
    total = 0
    device = next(model.parameters()).device
    
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
    acc = 100. * correct / total
    print(f"📊 Performance sur {name} : {acc:.2f}%")
    return acc

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# --- 3. INITIALISATION ET BASELINE ---

# Initialisation
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GestureClassifier().to(device)

# Affichage du nombre de paramètres (Demandé Point 3 du PDF)
print(f"🔢 Nombre de paramètres entraînables : {count_parameters(model):,}")

# Préparation des loaders
train_A_loader = DataLoader(train_A_ds, batch_size=32, shuffle=True)
test_A_loader  = DataLoader(test_A_ds, batch_size=32, shuffle=False)
test_B_loader  = DataLoader(test_B_ds, batch_size=32, shuffle=False)

# Optimiseur ADAMW (Exigence Prof)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

# --- 4. ENTRAÎNEMENT BASELINE (DOMAINE A UNIQUEMENT) ---
print("\n" + "="*50)
print("PHASE 1 : ENTRAÎNEMENT BASELINE (DOMAINE A)")
print("="*50)

model = train_model(model, train_A_loader, criterion, optimizer, epochs=5, device=device)

# Sauvegarde du modèle de base (il servira de point de départ pour toutes les expériences)
baseline_path = "resnet18_baseline.pth"
torch.save(model.state_dict(), baseline_path)
print(f"💾 Modèle de base sauvegardé : {baseline_path}")

# --- 5. ÉVALUATION INITIALE (DIAGNOSTIC) ---
print("\n" + "="*50)
print("PHASE 2 : ÉVALUATION INITIALE (AVANT ACTIVE LEARNING)")
print("="*50)

acc_A = evaluate_model(model, test_A_loader, "Domaine A (Kaggle)")
acc_B = evaluate_model(model, test_B_loader, "Domaine B (Webcam)")

# Analyse automatique pour le rapport
if acc_A > acc_B + 20:
    print("\n💡 ANALYSE : Écart important détecté (Domain Gap).")
    print("   Le modèle est bon sur A mais mauvais sur B.")
    print("   C'est la situation idéale pour tester l'Active Learning !")
else:
    print("\n💡 ANALYSE : Les performances sont proches ou déjà élevées.")
    print("   L'Active Learning apportera moins de gain visible.")

# Sauvegarde des résultats pour le tableau du rapport
baseline_results = {
    "Acc_A": acc_A,
    "Acc_B": acc_B,
    "Time": time.time(),
    "Params": count_parameters(model)
}

🔢 Nombre de paramètres entraînables : 11,178,051

PHASE 1 : ENTRAÎNEMENT BASELINE (DOMAINE A)
🚀 Début de l'entraînement (5 époques)...
   Époque 1/5 | Loss: 0.3295 | Acc Train: 88.43%
   Époque 2/5 | Loss: 0.0172 | Acc Train: 99.71%
   Époque 3/5 | Loss: 0.0049 | Acc Train: 100.00%
   Époque 4/5 | Loss: 0.0038 | Acc Train: 100.00%
   Époque 5/5 | Loss: 0.0021 | Acc Train: 100.00%
✅ Entraînement terminé en 960.9s
💾 Modèle de base sauvegardé : resnet18_baseline.pth

PHASE 2 : ÉVALUATION INITIALE (AVANT ACTIVE LEARNING)
📊 Performance sur Domaine A (Kaggle) : 84.44%
📊 Performance sur Domaine B (Webcam) : 58.64%

💡 ANALYSE : Écart important détecté (Domain Gap).
   Le modèle est bon sur A mais mauvais sur B.
   C'est la situation idéale pour tester l'Active Learning !


In [6]:
# =================================================================
# BLOC 3 : STRATÉGIES DE SÉLECTION (CERVEAU DE L'AL)
# =================================================================

import numpy as np
import torch
from sklearn.metrics import pairwise_distances

# --- 1. STRATÉGIE ALÉATOIRE (RÉFÉRENCE) ---
def random_selection(model, pool_data, n_samples, device):
    """
    Sélectionne n_samples au hasard dans le pool.
    C'est la baseline à battre.
    """
    n_pool = len(pool_data)
    indices = np.random.choice(n_pool, n_samples, replace=False)
    return indices

# --- 2. STRATÉGIE BASÉE SUR L'INCERTITUDE (MARGIN SAMPLING) ---
def uncertainty_selection(model, pool_data, n_samples, device):
    """
    Sélectionne les images où le modèle hésite le plus 
    (différence entre les 2 meilleures probabilités est faible).
    """
    model.eval()
    pool_loader = DataLoader(pool_data, batch_size=64, shuffle=False)
    all_margins = []
    
    with torch.no_grad():
        for inputs, _ in pool_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            probs = torch.nn.functional.softmax(outputs, dim=1)
            
            # On trie les probabilités
            probs_sorted, _ = torch.sort(probs, descending=True)
            
            # Marge = Top1 - Top2. Si proche de 0, le modèle hésite.
            margins = probs_sorted[:, 0] - probs_sorted[:, 1]
            all_margins.extend(margins.cpu().numpy())
            
    # On prend les indices des marges les plus petites
    indices = np.argsort(all_margins)[:n_samples]
    return indices

# --- 3. STRATÉGIE BASÉE SUR LA DIVERSITÉ (CORESET) ---
def get_embeddings(model, dataset, device):
    """
    Extrait les features internes du modèle (avant la couche finale).
    Nécessaire pour calculer la diversité.
    """
    model.eval()
    loader = DataLoader(dataset, batch_size=64, shuffle=False)
    embeddings = []
    
    with torch.no_grad():
        for inputs, _ in loader:
            inputs = inputs.to(device)
            # Astuce : on prend la sortie de la couche 'avgpool' de ResNet
            # C'est le vecteur de caractéristiques compressé
            x = model.resnet.conv1(inputs)
            x = model.resnet.bn1(x)
            x = model.resnet.relu(x)
            x = model.resnet.maxpool(x)
            x = model.resnet.layer1(x)
            x = model.resnet.layer2(x)
            x = model.resnet.layer3(x)
            x = model.resnet.layer4(x)
            x = model.resnet.avgpool(x)
            x = torch.flatten(x, 1)
            embeddings.append(x.cpu().numpy())
            
    return np.vstack(embeddings)

def diversity_selection(model, pool_data, n_samples, device):
    """
    Sélectionne des images différentes entre elles (éloignées dans l'espace des features).
    Utilise l'algorithme K-Center Greedy.
    """
    # 1. Obtenir les features
    embeddings = get_embeddings(model, pool_data, device)
    
    # 2. K-Center Greedy simplifié
    # On commence par prendre un point au hasard, puis on prend le plus loin, etc.
    # Pour simplifier ici, on utilise une méthode de sous-échantillonnage distant
    
    # On choisit le premier au hasard
    selected_indices = [np.random.randint(0, len(embeddings))]
    
    for _ in range(n_samples - 1):
        # Calculer la distance de tous les points aux points déjà sélectionnés
        # On garde la distance minimale de chaque point au set sélectionné
        selected_embs = embeddings[selected_indices]
        
        # Distance de chaque point du pool aux points sélectionnés
        dists = pairwise_distances(embeddings, selected_embs, metric='euclidean')
        min_dists = np.min(dists, axis=1)
        
        # Le point le plus éloigné est le plus "divers"
        new_idx = np.argmax(min_dists)
        selected_indices.append(new_idx)
        
    return np.array(selected_indices)

# --- 4. STRATÉGIE MIXTE (COMBINAISON) ---
def hybrid_selection(model, pool_data, n_samples, device):
    """
    Combine Incertitude et Diversité.
    1. Filtre les 20% les plus incertains.
    2. Parmi eux, prend les plus divers.
    """
    n_pool = len(pool_data)
    n_uncertainty = int(n_pool * 0.2) # On prend les 20% plus incertains
    
    # Étape 1 : Trouver les indices incertains
    uncertain_indices = uncertainty_selection(model, pool_data, n_uncertainty, device)
    
    # Étape 2 : Calculer la diversité parmi ces incertains
    # On va utiliser K-Center uniquement sur ce sous-ensemble
    
    # Créer un sous-ensemble des données pour extraire les embeddings
    subset_indices = uncertain_indices
    
    # On réutilise la logique de diversité mais sur le subset
    embeddings = get_embeddings(model, pool_data, device)
    subset_embs = embeddings[subset_indices]
    
    # Greedy K-Center sur le subset
    # On prend un point au hasard dans le subset pour commencer
    local_selected = [0] 
    for _ in range(n_samples - 1):
        selected_embs = subset_embs[local_selected]
        dists = pairwise_distances(subset_embs, selected_embs)
        min_dists = np.min(dists, axis=1)
        # On interdit de reprendre les mêmes (distance 0)
        min_dists[local_selected] = -1 
        new_idx = np.argmax(min_dists)
        local_selected.append(new_idx)
        
    # On map les indices locaux du subset vers les indices globaux du pool
    final_indices = subset_indices[local_selected]
    
    return final_indices

print("✅ Toutes les stratégies de sélection sont définies.")
print("   - Random")
print("   - Uncertainty (Margin)")
print("   - Diversity (Coreset)")
print("   - Hybrid (Uncertainty + Diversity)")

✅ Toutes les stratégies de sélection sont définies.
   - Random
   - Uncertainty (Margin)
   - Diversity (Coreset)
   - Hybrid (Uncertainty + Diversity)


In [ ]:
# =================================================================
# BLOC 4 : BOUCLE D'EXPÉRIENCE PRINCIPALE (AVEC REPRISE)
# =================================================================

import pandas as pd
import numpy as np
import torch
import os
from torch.utils.data import DataLoader

# --- PARAMÈTRES DE L'EXPÉRIENCE ---
BUDGET_RATIOS = [0.01, 0.02, 0.05, 0.10, 0.20, 0.50]
N_RUNS = 3  # Nombre de répétitions
EPOCHS_AL = 3 # Époques pour l'adaptation

# --- DÉFINITION DE LA 2ÈME STRATÉGIE MIXTE (MANQUANTE) ---
def weighted_hybrid_selection(model, pool_data, n_samples, device):
    """
    Stratégie Mixte 2 : Score pondéré.
    Score = (Poids Incertitude * Score Incertitude) + (Poids Diversité * Score Diversité)
    """
    model.eval()
    pool_loader = DataLoader(pool_data, batch_size=64, shuffle=False)
    all_uncertainties = []
    
    # 1. Calcul Incertitude
    with torch.no_grad():
        for inputs, _ in pool_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            probs = torch.nn.functional.softmax(outputs, dim=1)
            probs_sorted, _ = torch.sort(probs, descending=True)
            margins = probs_sorted[:, 0] - probs_sorted[:, 1]
            scores = 1.0 - margins.cpu().numpy() # Petit marge = gros score
            all_uncertainties.extend(scores)
            
    uncertainty_scores = np.array(all_uncertainties)
    
    # 2. Calcul Diversité (Distance au centre du pool)
    embeddings = get_embeddings(model, pool_data, device)
    center = np.mean(embeddings, axis=0)
    dists = np.linalg.norm(embeddings - center, axis=1)
    diversity_scores = (dists - dists.min()) / (dists.max() - dists.min() + 1e-6)
    
    # 3. Combinaison
    final_scores = 0.5 * uncertainty_scores + 0.5 * diversity_scores
    indices = np.argsort(final_scores)[-n_samples:]
    return indices

# --- LISTE DES STRATÉGIES (MISE À JOUR) ---
strategies = {
    "Random": random_selection,
    "Uncertainty": uncertainty_selection,
    "Diversity": diversity_selection,
    "Hybrid": hybrid_selection,            # Mixte 1 (Filtrage)
    "Weighted_Hybrid": weighted_hybrid_selection # Mixte 2 (Pondération)
}

# Fichier de sauvegarde
csv_file = "active_learning_results.csv"

# --- SYSTÈME DE REPRISE (Pour ne pas tout recommencer) ---
# Si le fichier existe déjà, on charge ce qui a été fait
if os.path.exists(csv_file):
    df_existing = pd.read_csv(csv_file)
    print(f"📂 Fichier existant détecté. {len(df_existing)} entrées déjà calculées.")
else:
    # Création du fichier avec l'en-tête si nouveau
    with open(csv_file, "w") as f:
        f.write("Strategy,Run,Budget,Accuracy_B\n")
    df_existing = pd.DataFrame(columns=["Strategy", "Run", "Budget", "Accuracy_B"])

print(f"🚧 DÉBUT / REPRISE DES EXPÉRIENCES.")
print(f"⏳ Temps estimé : Long (CPU). Patience...")

# --- BOUCLE PRINCIPALE ---

for strategy_name, strategy_func in strategies.items():
    print(f"\n{'='*60}")
    print(f"STRATÉGIE EN COURS : {strategy_name}")
    print(f"{'='*60}")
    
    for run in range(1, N_RUNS + 1):
        # Vérification rapide : ce run est-il déjà fini ?
        # On regarde si la dernière ligne de ce run (Budget 0.5) est dans le CSV
        mask = (df_existing["Strategy"] == strategy_name) & (df_existing["Run"] == run) & (df_existing["Budget"] == 0.50)
        
        if not df_existing.empty and mask.any():
            print(f"   ✅ Run {run}/{N_RUNS} déjà effectué. Passage au suivant.")
            continue
            
        print(f"\n--- Run {run}/{N_RUNS} pour {strategy_name} ---")
        
        # 1. RÉINITIALISATION DU MODÈLE
        model.load_state_dict(torch.load("resnet18_baseline.pth"))
        model.to(device)
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
        
        # 2. RÉINITIALISATION DES DONNÉES
        X_train_current = X_train_A.copy()
        y_train_current = y_train_A.copy()
        
        X_pool_current = X_pool_B.copy()
        y_pool_current = y_pool_B.copy()
        
        previous_budget_count = 0
        
        # 3. BOUCLE SUR LES BUDGETS
        for ratio in BUDGET_RATIOS:
            
            # Vérification : Ce point précis est-il déjà dans le CSV ?
            point_mask = (df_existing["Strategy"] == strategy_name) & (df_existing["Run"] == run) & (df_existing["Budget"] == ratio)
            
            if not df_existing.empty and point_mask.any():
                # Si le point existe, on met juste à jour les variables pour continuer la suite
                # On charge l'accuracy enregistrée juste pour info
                saved_acc = df_existing[point_mask]["Accuracy_B"].values[0]
                print(f"  -> Budget {ratio*100:.0f}% déjà fait (Acc: {saved_acc:.2f}%). On continue...")
                
                # IMPORTANT : Il faut remettre le compte à jour pour que le prochain step soit correct
                previous_budget_count = int(len(X_pool_B) * ratio)
                continue

            # Calcul du nombre d'images à prélever
            target_count = int(len(X_pool_B) * ratio)
            n_query = target_count - previous_budget_count
            
            if n_query <= 0: continue
            
            print(f"  -> Budget {ratio*100:.0f}% : Sélection de {n_query} images...")
            
            # A. SÉLECTION
            temp_pool_ds = to_tensor(X_pool_current, y_pool_current)
            indices = strategy_func(model, temp_pool_ds, n_query, device)
            
            # B. MISE À JOUR
            X_selected = X_pool_current[indices]
            y_selected = y_pool_current[indices]
            
            X_train_current = np.concatenate([X_train_current, X_selected])
            y_train_current = np.concatenate([y_train_current, y_selected])
            
            X_pool_current = np.delete(X_pool_current, indices, axis=0)
            y_pool_current = np.delete(y_pool_current, indices, axis=0)
            
            previous_budget_count = target_count
            
            # C. ENTRAÎNEMENT
            train_loader = DataLoader(to_tensor(X_train_current, y_train_current), batch_size=32, shuffle=True)
            model = train_model(model, train_loader, criterion, optimizer, epochs=EPOCHS_AL, device=device)
            
            # D. ÉVALUATION
            acc_B = evaluate_model(model, test_B_loader, name=f"{strategy_name} Run{run}")
            
            # E. SAUVEGARDE
            with open(csv_file, "a") as f:
                f.write(f"{strategy_name},{run},{ratio},{acc_B}\n")
            
            # On recharge le dataframe pour que la vérification suivante soit à jour
            df_existing = pd.read_csv(csv_file)

print("\n" + "="*60)
print("✅ TOUTES LES EXPÉRIENCES SONT TERMINÉES.")
print(f"📊 Fichier de résultats final : {csv_file}")
print("="*60)

📂 Fichier existant détecté. 19 entrées déjà calculées.
🚧 DÉBUT / REPRISE DES EXPÉRIENCES.
⏳ Temps estimé : Long (CPU). Patience...

STRATÉGIE EN COURS : Random
   ✅ Run 1/3 déjà effectué. Passage au suivant.
   ✅ Run 2/3 déjà effectué. Passage au suivant.
   ✅ Run 3/3 déjà effectué. Passage au suivant.

STRATÉGIE EN COURS : Uncertainty

--- Run 1/3 pour Uncertainty ---
  -> Budget 1% déjà fait (Acc: 52.34%). On continue...
  -> Budget 2% : Sélection de 7 images...
🚀 Début de l'entraînement (3 époques)...
   Époque 1/3 | Loss: 0.0138 | Acc Train: 99.51%
   Époque 2/3 | Loss: 0.0338 | Acc Train: 98.83%
   Époque 3/3 | Loss: 0.0391 | Acc Train: 99.12%
✅ Entraînement terminé en 539.5s
📊 Performance sur Uncertainty Run1 : 50.89%
  -> Budget 5% : Sélection de 23 images...
🚀 Début de l'entraînement (3 époques)...
   Époque 1/3 | Loss: 0.0593 | Acc Train: 97.71%
   Époque 2/3 | Loss: 0.0102 | Acc Train: 99.71%
   Époque 3/3 | Loss: 0.0031 | Acc Train: 100.00%
✅ Entraînement terminé en 533.4s
📊

In [ ]:
# =================================================================
# BLOC 5 : GÉNÉRATION DES COURBES ET ANALYSE FINALE
# =================================================================

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# --- 1. CHARGEMENT DES DONNÉES ---
csv_file = "active_learning_results.csv"
df = pd.read_csv(csv_file)

print(f"📊 Données chargées : {len(df)} entrées trouvées.")

# --- 2. CALCUL DES MOYENNES ET INTERVALLES DE CONFIANCE ---
# On groupe par Stratégie et Budget pour calculer la moyenne et l'écart-type
summary = df.groupby(['Strategy', 'Budget'])['Accuracy_B'].agg(['mean', 'std', 'count']).reset_index()

# Calcul de l'intervalle de confiance (95%)
# Formule : 1.96 * (écart-type / sqrt(nombre_de_runs))
summary['ci'] = 1.96 * (summary['std'] / np.sqrt(summary['count']))

# --- 3. PARAMÈTRES D'AFFICHAGE (STYLE PROFESSIONNEL) ---
plt.figure(figsize=(10, 7))
sns.set_theme(style="whitegrid") # Style scientifique propre

# Palette de couleurs pour distinguer les stratégies
colors = {
    "Random": "gray",
    "Uncertainty": "blue",
    "Diversity": "orange",
    "Hybrid": "green",
    "Weighted_Hybrid": "red"
}

# --- 4. TRAÇAGE DES COURBES ---
strategies_to_plot = summary['Strategy'].unique()

for strat in strategies_to_plot:
    data = summary[summary['Strategy'] == strat]
    
    # Tracer la ligne principale (Moyenne)
    plt.plot(data['Budget'] * 100, data['mean'], 
             marker='o', 
             label=strat, 
             color=colors.get(strat, 'black'),
             linewidth=2)
    
    # Tracer la "zone d'ombre" (Intervalle de confiance)
    plt.fill_between(data['Budget'] * 100, 
                     data['mean'] - data['ci'], 
                     data['mean'] + data['ci'], 
                     color=colors.get(strat, 'black'), 
                     alpha=0.2) # Transparence

# --- 5. MISE EN FORME (AXES, TITRES, LÉGENDE) ---
plt.title("Performance du modèle sur le Domaine B vs Coût d'annotation", fontsize=14, fontweight='bold')
plt.xlabel("Budget d'annotation (%)", fontsize=12)
plt.ylabel("Accuracy sur Test B (%)", fontsize=12)
plt.legend(title="Stratégies", loc='lower right', fontsize=10)

# Forcer l'axe X à afficher les pourcentages définis
plt.xticks([1, 2, 5, 10, 20, 50]) 
plt.ylim(bottom=min(summary['mean']) - 5, top=100) # Espace pour bien voir

# Grille pour faciliter la lecture
plt.grid(True, linestyle='--', alpha=0.7)

# --- 6. SAUVEGARDE ET AFFICHAGE ---
plot_filename = "active_learning_curves.png"
plt.savefig(plot_filename, dpi=300, bbox_inches='tight')
print(f"📸 Courbe sauvegardée sous : {plot_filename}")

plt.show()

# --- 7. ANALYSE AUTOMATIQUE POUR LE RAPPORT ---
print("\n" + "="*60)
print("📋 SYNTHÈSE DES RÉSULTATS (À COPIER DANS LE RAPPORT)")
print("="*60)

# Performance initiale (Budget 1%) vs Finale (Budget 50%)
for strat in strategies_to_plot:
    data_strat = summary[summary['Strategy'] == strat]
    if len(data_strat) > 0:
        first_point = data_strat.iloc[0]['mean']
        last_point = data_strat.iloc[-1]['mean']
        print(f"🔹 {strat} : Départ {first_point:.2f}% -> Fin {last_point:.2f}% (Gain: +{last_point-first_point:.2f}%)")

print("\n💡 Analyse des stratégies :")
best_start_strat = summary[summary['Budget'] == 0.01].sort_values(by='mean', ascending=False).iloc[0]['Strategy']
best_end_strat = summary[summary['Budget'] == 0.50].sort_values(by='mean', ascending=False).iloc[0]['Strategy']

print(f"   - Meilleure performance à faible budget (1%) : {best_start_strat}")
print(f"   - Meilleure performance à fort budget (50%) : {best_end_strat}")
print("="*60)

In [ ]:
# =================================================================
# BLOC 6 : ANALYSE QUALITATIVE (VISUALISATION DES CHOIX)
# =================================================================

import matplotlib.pyplot as plt
import random

def show_selected_images(X_pool, indices, title):
    """
    Affiche une grille d'images sélectionnées par une stratégie.
    """
    plt.figure(figsize=(10, 5))
    plt.suptitle(title, fontsize=16, fontweight='bold')
    
    # On prend jusqu'à 9 images max pour l'affichage
    display_indices = indices[:9]
    
    for i, idx in enumerate(display_indices):
        plt.subplot(3, 3, i+1)
        
        # Revenir au format image (H, W, C) et inverser la normalisation pour l'affichage
        img = X_pool[idx]
        
        # Image numpy (H, W, C) - Remettre les valeurs entre 0 et 255 pour affichage correct
        # Tes images X_pool sont déjà en 0-255 RGB normalement
        plt.imshow(img.astype(int))
        plt.axis('off')
        
    plt.tight_layout()
    plt.show()

# --- 1. Préparation : On prend le modèle Baseline pour simuler une sélection ---
model.load_state_dict(torch.load("resnet18_baseline.pth"))
model.eval()

# On travaille sur le Pool B complet (non annoté au début)
X_pool_full = X_pool_B.copy()
y_pool_full = y_pool_B.copy()
pool_dataset = to_tensor(X_pool_full, y_pool_full)

# Nombre d'images à sélectionner pour l'exemple
N_VISU = 50

# --- 2. SÉLECTION PAR INCERTITUDE ---
print("🔍 Sélection des images les plus INCERTAINES...")
indices_uncertainty = uncertainty_selection(model, pool_dataset, N_VISU, device)
show_selected_images(X_pool_full, indices_uncertainty, "Images sélectionnées : Incertitude (Modèle hésite)")

# --- 3. SÉLECTION PAR DIVERSITÉ ---
print("🔍 Sélection des images les plus DIVERSES...")
indices_diversity = diversity_selection(model, pool_dataset, N_VISU, device)
show_selected_images(X_pool_full, indices_diversity, "Images sélectionnées : Diversité (Couverture spatiale)")

# --- 4. SÉLECTION ALÉATOIRE (Comparaison) ---
print("🔍 Sélection aléatoire (Témoin)...")
indices_random = random.sample(range(len(X_pool_full)), N_VISU)
show_selected_images(X_pool_full, indices_random, "Images sélectionnées : Aléatoire")

print("\n✅ Analyse qualitative terminée.")

In [ ]:
# =================================================================
# ÉTAPE 3 : ARCHITECTURE ET STRATÉGIE D'ACTIVE LEARNING
# =================================================================

def get_resnet_model():
    """
    Configure un ResNet-18 pré-entraîné.
    Section 5 du rapport : "Architecture ResNet-18 (ImageNet)".
    """
    # 1. Chargement du modèle avec les poids optimaux de PyTorch
    model = torch.hub.load('pytorch/vision:v0.10.0', 'resnet18', weights='ResNet18_Weights.DEFAULT')
    
    # 2. Adaptation de la couche de sortie
    # ResNet-18 sort normalement 1000 classes (ImageNet). 
    # On le modifie pour tes 3 classes : Pierre, Feuille, Ciseaux.
    num_ftrs = model.fc.in_features
    model.fc = torch.nn.Linear(num_ftrs, 3) 
    
    return model



def margin_confidence(net, pool_dataset, n=20):
    """
    Stratégie d'Incertitude : Margin Sampling.
    Sélectionne les n images où le modèle hésite le plus entre deux classes.
    """
    net.eval()
    all_margins = []
    device = next(net.parameters()).device
    
    # DataLoader temporaire pour parcourir le pool d'images non annotées
    loader = DataLoader(pool_dataset, batch_size=32, shuffle=False)
    
    with torch.no_grad():
        for batch_data, _ in loader: # On ignore les labels car on simule qu'ils sont inconnus
            batch_data = batch_data.to(device)
            outputs = net(batch_data)
            
            # Conversion des scores en probabilités (0 à 1)
            probs = torch.nn.functional.softmax(outputs, dim=1)
            
            # On trie les probabilités par ordre décroissant
            # probs_sorted[:, 0] est la probabilité la plus haute
            # probs_sorted[:, 1] est la deuxième plus haute
            probs_sorted, _ = torch.sort(probs, descending=True, dim=1)
            
            # Calcul de la marge (Différence entre les deux meilleurs choix)
            # Plus la marge est proche de 0, plus l'incertitude est grande
            margins = probs_sorted[:, 0] - probs_sorted[:, 1]
            all_margins.extend(margins.cpu().numpy())
            
    # np.argsort trie du plus petit au plus grand.
    # On prend les 'n' premiers indices (les marges les plus petites = incertitude max)
    uncertain_indices = np.argsort(all_margins)[:n]
    
    return uncertain_indices

✅ Modèle, Entraînement et Stratégies définis avec succès.


In [ ]:
# =================================================================
# ÉTAPE 3 & 5 : ENTRAÎNEMENT DU MODÈLE DE RÉFÉRENCE (BASELINE)
# =================================================================

# 1. Préparation du DataLoader pour le Domaine A (Kaggle)
# On utilise un batch_size de 32 comme indiqué dans le tableau 3 du rapport
train_loader_A = DataLoader(train_A_ds, batch_size=32, shuffle=True)
test_loader_A  = DataLoader(test_A_ds, batch_size=32, shuffle=False)

# 2. Configuration des paramètres (Mise à jour vers AdamW)
criterion = torch.nn.CrossEntropyLoss()
# AdamW est plus performant pour le transfert learning avec ResNet
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

# 3. Lancement de l'entraînement initial
print("\n--- 🚀 ENTRAÎNEMENT DU MODÈLE DE BASE (DOMAINE A) ---")
# On entraîne sur quelques époques pour avoir une base solide
model = train_model(model, train_loader_A, criterion, optimizer, epochs=5, device=device)

# 4. Évaluation initiale sur le Domaine A et le Domaine B
def evaluate_baseline(model, loader, name):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    acc = 100. * correct / total
    print(f"📊 Accuracy sur {name} : {acc:.2f}%")
    return acc

# Test sur le Domaine A (devrait être élevé)
acc_A = evaluate_baseline(model, test_loader_A, "Domaine A (Kaggle)")

# Test sur le Domaine B (devrait être bas avant l'Active Learning)
test_loader_B = DataLoader(test_B_ds, batch_size=32, shuffle=False)
acc_B = evaluate_baseline(model, test_loader_B, "Domaine B (Webcam - Baseline)")

# --- 5. SAUVEGARDE DU MODÈLE ---
# On sauvegarde ce point de départ pour pouvoir comparer les stratégies plus tard
torch.save(model.state_dict(), "resnet_baseline.pth")
print("\n✅ Modèle baseline sauvegardé sous 'resnet_baseline.pth'")


--- ENTRAÎNEMENT DU MODÈLE DE BASE ---
🚀 Entraînement sur cpu pour 3 époques...
Époque 1/3 | Loss: 0.0248 | Acc: 99.61%
Époque 2/3 | Loss: 0.0066 | Acc: 99.90%
Époque 3/3 | Loss: 0.0036 | Acc: 100.00%
✅ Temps total : 576.6 secondes


In [ ]:
# =================================================================
# ÉTAPE 3 & 5 : BILAN DES DONNÉES ET ENTRAÎNEMENT BASELINE
# =================================================================

import pandas as pd
import numpy as np
import time
import os
from pathlib import Path
import torch
from torch.utils.data import DataLoader

# --- 1. CHARGEMENT ROBUSTE DES IMAGES DU DOMAINE B ---
# Cette fonction assure que l'on ne tente pas de lire les vidéos comme des images
def load_images_only_fixed(root_path, subset, current_classes, current_label_map):
    X, y = [], []
    subset_path = Path(root_path) / subset
    if not subset_path.exists(): 
        print(f"❌ Chemin introuvable : {subset_path}")
        return np.array(X), np.array(y)

    for cls_name in current_classes:
        cls_folder = subset_path / cls_name
        if not cls_folder.exists(): continue
        
        for file_name in os.listdir(cls_folder):
            # Filtrage strict sur l'extension .jpg (images extraites)
            if file_name.lower().endswith('.jpg'):
                file_path = cls_folder / file_name
                img = cv2.imread(str(file_path))
                if img is not None:
                    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                    img = cv2.resize(img, (224, 224))
                    X.append(img)
                    y.append(current_label_map[cls_name])
    return np.array(X), np.array(y)

# Chargement effectif
X_pool_B, y_pool_B = load_images_only_fixed(PATH_DOMAIN_B, "pool", CLASSES_B, LABEL_MAP_B)
X_test_B, y_test_B = load_images_only_fixed(PATH_DOMAIN_B, "test", CLASSES_B, LABEL_MAP_B)

# Conversion en Tenseurs PyTorch via ta fonction to_tensor
pool_B_ds  = to_tensor(X_pool_B, y_pool_B)
test_B_ds  = to_tensor(X_test_B, y_test_B)

# --- 2. GÉNÉRATION DU TABLEAU DE RÉPARTITION (Requis par le Point 3 du PDF) ---
def get_distribution_fixed(y, current_label_map):
    dist = {}
    for name, label in current_label_map.items():
        count = (np.array(y) == label).sum()
        dist[name] = int(count) 
    return dist

dist_A = get_distribution_fixed(y_train_A, LABEL_MAP_A)
dist_B_pool = get_distribution_fixed(y_pool_B, LABEL_MAP_B)
dist_B_test = get_distribution_fixed(y_test_B, LABEL_MAP_B)

print("\n📊 TABLEAU DE RÉPARTITION DES DONNÉES")
df_stats = pd.DataFrame([dist_A, dist_B_pool, dist_B_test], 
                  index=["Domaine A (Train)", "Domaine B (Pool)", "Domaine B (Test)"])
print(df_stats.fillna(0).astype(int))

# --- 3. OPTIMISATION : FREEZING DU RÉSEAU ---
# On gèle les couches de base de ResNet pour ne pas les modifier
for param in model.parameters():
    param.requires_grad = False
# On ne garde que la dernière couche (FC) active pour l'entraînement
for param in model.resnet.fc.parameters():
    param.requires_grad = True

# --- 4. ENTRAÎNEMENT DE LA BASELINE ---
print("\n🚀 Lancement de l'entraînement Baseline (Domaine A)...")
criterion = torch.nn.CrossEntropyLoss()
# Utilisation de AdamW comme spécifié dans votre proposal
optimizer = torch.optim.AdamW(model.resnet.fc.parameters(), lr=1e-3)
train_loader = DataLoader(train_A_ds, batch_size=64, shuffle=True)

start_time = time.time()
# On utilise ta fonction train_model définie plus haut
model = train_model(model, train_loader, criterion, optimizer, epochs=3, device=device)
execution_time = time.time() - start_time

# --- 5. ÉVALUATION FINALE DE L'ÉTAPE 3 ---
def get_accuracy_final(net, loader):
    net.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            preds = net(X)
            correct += (preds.argmax(1) == y).sum().item()
            total += y.size(0)
    return (100 * correct / total) if total > 0 else 0

test_A_loader = DataLoader(test_A_ds, batch_size=64)
test_B_loader = DataLoader(test_B_ds, batch_size=64)

acc_A = get_accuracy_final(model, test_A_loader)
acc_B = get_accuracy_final(model, test_B_loader)

print("-" * 50)
print(f"📈 Performance Initiale sur Domaine A : {acc_A:.2f}%")
print(f"📉 Performance Initiale sur Domaine B : {acc_B:.2f}%")
print(f"⏱️ Temps de calcul : {execution_time:.2f}s")
print("-" * 50)

📂 Chargement des images du Domaine B (pool et test)...

📊 TABLEAU DE RÉPARTITION RÉEL
                  rock  paper  scissors  0_Pierre  1_Feuille  2_Ciseaux
Train A (Kaggle)   329    362       329         0          0          0
pool B (Actif)       0      0         0        48         49         36
test B (Final)       0      0         0        51         57         57

🚀 Lancement de l'entraînement Baseline (3 époques)...
🚀 Entraînement sur cpu pour 3 époques...
Époque 1/3 | Loss: 0.0020 | Acc: 99.90%
Époque 2/3 | Loss: 0.0004 | Acc: 100.00%
Époque 3/3 | Loss: 0.0017 | Acc: 99.90%
✅ Temps total : 450.0 secondes

⚖️ ÉVALUATION DES DOMAINES...
--------------------------------------------------
✅ Accuracy Domaine A (Kaggle) : 82.04%
✅ Accuracy Domaine B (Moi)    : 64.85%
⏱️ Temps total de calcul      : 449.99s
--------------------------------------------------


In [ ]:
# =================================================================
# ÉTAPE 4 : LA BOUCLE D'APPRENTISSAGE ACTIF (ACTIVE LEARNING)
# =================================================================

# 1. Préparation des jeux de données de travail
# On part du modèle entraîné sur A, et on va lui injecter des données de B
X_train_combined = X_train_A.copy()
y_train_combined = y_train_A.copy()

# On garde une copie du pool B pour piocher dedans
X_pool_current = X_pool_B.copy()
y_pool_current = y_pool_B.copy()

# --- PARAMÈTRES DE LA BOUCLE ---
# Dans ton projet, on simule des itérations (ex: 4 cycles)
# À chaque cycle, on demande au modèle de choisir 20 images (N_QUERY)
N_ITERATIONS = 4 
N_QUERY = 20 

print(f"🚀 Démarrage de la boucle : {N_ITERATIONS} itérations, {N_QUERY} images par cycle.")

for i in range(1, N_ITERATIONS + 1):
    print(f"\n--- 🔄 CYCLE D'ANNOTATION {i}/{N_ITERATIONS} ---")
    
    # A. Création d'un dataset temporaire pour le calcul de l'incertitude
    # On transforme le pool actuel en tenseurs pour que le modèle puisse les lire
    temp_pool_ds = to_tensor(X_pool_current, y_pool_current)
    
    # B. SÉLECTION (STRATÉGIE DE MARGE)
    # On utilise la fonction margin_confidence définie précédemment
    indices_a_annoter = margin_confidence(model, temp_pool_ds, n=N_QUERY)
    
    # C. "ANNOTATION" (On récupère les images et leurs vrais labels)
    X_to_add = X_pool_current[indices_a_annoter]
    y_to_add = y_pool_current[indices_a_annoter]
    
    # D. MISE À JOUR DU JEU D'ENTRAÎNEMENT
    X_train_combined = np.concatenate([X_train_combined, X_to_add])
    y_train_combined = np.concatenate([y_train_combined, y_to_add])
    
    # E. NETTOYAGE DU POOL (On retire les images déjà choisies)
    X_pool_current = np.delete(X_pool_current, indices_a_annoter, axis=0)
    y_pool_current = np.delete(y_pool_current, indices_a_annoter, axis=0)
    
    # F. RÉ-ENTRAÎNEMENT (TRANSFER LEARNING)
    # On entraîne sur le mélange (Domaine A + Échantillons choisis de B)
    # 2 époques suffisent pour ajuster le modèle sans tout oublier
    combined_loader = DataLoader(to_tensor(X_train_combined, y_train_combined), 
                                 batch_size=32, shuffle=True)
    
    # On utilise AdamW comme dans ton proposal
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4) 
    model = train_model(model, combined_loader, criterion, optimizer, epochs=2, device=device)
    
    # G. ÉVALUATION (Pour suivre la progression)
    acc_A = get_accuracy(model, test_A_loader)
    acc_B = get_accuracy(model, test_B_loader)
    
    # H. ENREGISTREMENT DANS L'HISTORIQUE
    # On utilise la fonction add_to_history préparée précédemment
    current_budget = i * N_QUERY
    add_to_history(f"+{current_budget} imgs", len(X_train_combined), 0, 
                   len(X_test_B), len(X_pool_current), acc_A, acc_B)

print("\n✅ Boucle d'apprentissage actif terminée avec succès.")

In [ ]:
# # =================================================================
# # VARIANTE : CYCLE D'APPRENTISSAGE ACTIF AVEC BARRES DE PROGRESSION
# # =================================================================

# from tqdm.notebook import tqdm
# import time

# def train_fast(net, train_dataloader, criterion, optimizer, epochs=1, device='cpu'):
#     """
#     Version améliorée de la fonction train avec barre de progression tqdm.
#     """
#     for epoch in range(1, epochs+1):
#         net.train()
#         train_loss, train_accuracy, total_samples = 0.0, 0.0, 0
        
#         # Barre de progression interactive
#         with tqdm(train_dataloader, unit="batch", desc=f"Époque {epoch}/{epochs}", leave=True) as tepoch:
#             for X, y in tepoch:
#                 X, y = X.to(device), y.to(device)
                
#                 optimizer.zero_grad()
#                 preds = net(X)
#                 loss = criterion(preds, y)
#                 loss.backward()
#                 optimizer.step()

#                 # Mise à jour des métriques en temps réel
#                 train_loss += loss.item() * len(y)
#                 train_accuracy += (preds.argmax(1) == y).sum().item()
#                 total_samples += len(y)
                
#                 tepoch.set_postfix(loss=train_loss/total_samples, acc=100.*train_accuracy/total_samples)
#     return net

# # --- INITIALISATION DES VARIABLES DE TRAVAIL ---
# current_pool_X = X_pool_B.copy()
# current_pool_y = y_pool_B.copy()
# X_train_B_added, y_train_B_added = [], []
# results_evolution = []
# BUDGET_PER_STEP = 20 # Nombre d'images par itération
# ITERATIONS = 4



# # --- BOUCLE PRINCIPALE ---
# for it in range(ITERATIONS):
#     print(f"\n🔄 CYCLE D'ADAPTATION {it+1}/{ITERATIONS}")

#     # 1. Sélection par Margin Confidence
#     # On crée un dataset temporaire pour le pool actuel
#     temp_pool_ds = to_tensor(current_pool_X, current_pool_y)
#     selected_indices = margin_confidence(model, temp_pool_ds, n=BUDGET_PER_STEP)

#     # 2. Extraction des données sélectionnées
#     X_to_add = current_pool_X[selected_indices]
#     y_to_add = current_pool_y[selected_indices]
    
#     X_train_B_added.extend(X_to_add)
#     y_train_B_added.extend(y_to_add)

#     # 3. Mise à jour du pool (on retire ce qu'on a pris)
#     current_pool_X = np.delete(current_pool_X, selected_indices, axis=0)
#     current_pool_y = np.delete(current_pool_y, selected_indices, axis=0)

#     # 4. Entraînement combiné (A + B sélectionné)
#     X_comb = np.concatenate([X_train_A, np.array(X_train_B_added)])
#     y_comb = np.concatenate([y_train_A, np.array(y_train_B_added)])
    
#     combined_loader = DataLoader(to_tensor(X_comb, y_comb), batch_size=32, shuffle=True)
    
#     # Fine-tuning rapide (1 époque suffit souvent en Active Learning)
#     model = train_fast(model, combined_loader, criterion, optimizer, epochs=1, device=device)

#     # 5. Évaluation et stockage
#     acc_b = get_accuracy(model, test_B_loader)
#     results_evolution.append(acc_b)
#     print(f"✅ Accuracy sur Domaine B après ajout : {acc_b:.2f}%")

# # --- VISUALISATION RAPIDE ---
# plt.plot(range(1, ITERATIONS+1), results_evolution, marker='o', color='orange')
# plt.title("Progression rapide de l'adaptation")
# plt.xlabel("Itérations")
# plt.ylabel("Accuracy B (%)")
# plt.grid(True)
# plt.show()

In [ ]:
# =================================================================
# ÉTAPE 5 & 6 : ARCHITECTURE DU MODÈLE ET STRATÉGIE D'INCERTITUDE
# =================================================================

import torch
import torch.nn as nn
import torchvision.models as models
import time
import numpy as np

# --- 1. DÉFINITION DU MODÈLE (Adapté pour ResNet-18 PyTorch standard) ---
class GestureClassifier(nn.Module):
    def __init__(self):
        super(GestureClassifier, self).__init__()
        # On charge le ResNet-18 standard (plus simple à gérer avec tes tenseurs)
        self.resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        
        # On remplace la dernière couche (fc) pour tes 3 classes (Pierre, Feuille, Ciseaux)
        num_ftrs = self.resnet.fc.in_features
        self.resnet.fc = nn.Linear(num_ftrs, 3) 

    def forward(self, x):
        # x est un tenseur de forme (Batch, 3, 224, 224)
        return self.resnet(x)

# --- 2. FONCTION D'ENTRAÎNEMENT (Optimisée AdamW) ---
def train_model(model, train_loader, criterion, optimizer, epochs=3, device='cpu'):
    model.to(device)
    start_time = time.time()
    
    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
        epoch_acc = 100. * correct / total
        print(f"Époque {epoch}/{epochs} | Loss: {running_loss/total:.4f} | Acc: {epoch_acc:.2f}%")
        
    print(f"✅ Entraînement terminé en {time.time() - start_time:.1f}s")
    return model

# --- 3. STRATÉGIE D'INCERTITUDE : MARGIN SAMPLING (Étape 6 du Proposal) ---
def get_margin_indices(model, unlabeled_loader, n_samples, device):
    """
    Sélectionne les n images où la différence entre les deux meilleures 
    prédictions est la plus petite (incertitude maximale).
    """
    model.eval()
    all_margins = []
    
    with torch.no_grad():
        for i, (inputs, _) in enumerate(unlabeled_loader):
            inputs = inputs.to(device)
            outputs = model(inputs)
            probs = torch.nn.functional.softmax(outputs, dim=1)
            
            # On trie les probabilités pour avoir les deux meilleures
            top_probs, _ = torch.topk(probs, 2, dim=1)
            
            # Marge = Prob1 - Prob2
            margins = top_probs[:, 0] - top_probs[:, 1]
            
            # On stocke la marge et l'index original
            for j, m in enumerate(margins):
                # i*batch_size + j donne l'index absolu dans le pool
                all_margins.append(m.item())
                
    # On récupère les indices des plus PETITES marges
    uncertain_indices = np.argsort(all_margins)[:n_samples]
    return uncertain_indices

# --- 4. INITIALISATION DU MODÈLE ET OPTIMISEUR ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GestureClassifier().to(device)

# Utilisation de AdamW comme spécifié dans ton Proposal
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
criterion = torch.nn.CrossEntropyLoss()

print("✅ Modèle et fonctions de sélection prêts.")

In [ ]:
# =================================================================
# ÉTAPE 6 : PRÉPARATION DU TABLEAU DE SUIVI (LOGGING)
# =================================================================

import pandas as pd

# Initialisation de la liste pour stocker l'évolution du projet
# Cela permettra de tracer les courbes de performance finales
history_data = []

def add_to_history(iteration, train_size, val_size, test_size, unlabeled_size, acc_A, acc_B):
    """
    Enregistre les statistiques à chaque étape du budget d'annotation.
    Utile pour prouver l'efficacité de l'Active Learning.
    """
    history_data.append({
        "Iteration": iteration,
        "Train (A+B)": train_size,       # Total des images utilisées pour l'entraînement
        "Validation": val_size,
        "Test B": test_size,             # Évaluation sur tes images
        "Unlabeled B": unlabeled_size,   # Images restantes non annotées
        "Acc Domaine A": f"{acc_A:.2f}%",
        "Acc Domaine B": f"{acc_B:.2f}%"
    })

# --- ENREGISTREMENT DE L'ÉTAT INITIAL (BASELINE) ---
# On commence par noter les performances sans avoir ajouté aucune image du Domaine B
add_to_history(
    "Initial (0%)", 
    len(X_train_A), 
    len(X_val_A) if 'X_val_A' in locals() else 0, 
    len(X_test_B), 
    len(X_pool_B), 
    acc_A, 
    acc_B
)

# Affichage du tableau de bord initial
df_iterations = pd.DataFrame(history_data)
print("\n📊 TABLEAU DE SUIVI DES ITÉRATIONS (Format Projet)")
print("="*70)
print(df_iterations.to_string(index=False))
print("="*70)

print("\n💡 Note : Ce tableau va se remplir à chaque fois qu'on ajoute 1%, 2%, 5%... du Domaine B.")

In [ ]:
# =================================================================
# ÉVALUATION : COMPARAISON DES PERFORMANCES (KAGGLE vs WEBCAM)
# =================================================================

def get_accuracy(model, dataloader):
    """
    Calcule la précision du modèle sur un chargeur de données spécifique.
    C'est le juge de paix pour mesurer l'efficacité de l'Active Learning.
    """
    model.eval() # Désactive le dropout et la normalisation de batch pour le test
    correct = 0
    total = 0
    device = next(model.parameters()).device # Détecte automatiquement si GPU ou CPU
    
    with torch.no_grad(): # Pas de calcul de gradient (gain de mémoire et vitesse)
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            outputs = model(X)
            # On prend l'index de la probabilité la plus élevée (0, 1 ou 2)
            _, predicted = torch.max(outputs.data, 1)
            total += y.size(0)
            correct += (predicted == y).sum().item()
            
    return 100 * correct / total

# --- EXÉCUTION DU CALCUL ---
print("\n--- ⚖️ CALCUL DES PERFORMANCES INITIALES ---")

# 1. Préparation du loader pour le Domaine A (Kaggle)
test_A_loader = DataLoader(test_A_ds, batch_size=32, shuffle=False)

# 2. Vérification et préparation du loader pour le Domaine B (Tes images)
if test_B_ds is not None and len(test_B_ds) > 0:
    test_B_loader = DataLoader(test_B_ds, batch_size=32, shuffle=False)
    acc_B = get_accuracy(model, test_B_loader)
else:
    acc_B = 0.0
    print("⚠️ Attention : Le set de test B est vide ou non chargé !")

# 3. Calcul pour le Domaine A
acc_A = get_accuracy(model, test_A_loader)

# --- AFFICHAGE DES RÉSULTATS ---
print("-" * 50)
print(f"✅ Précision sur Domaine A (Référence Kaggle) : {acc_A:.2f}%")
print(f"📉 Précision sur Domaine B (Tes mains)      : {acc_B:.2f}%")
print("-" * 50)

if acc_A > acc_B + 20:
    print("💡 Analyse : L'écart est important. L'Active Learning est justifié !")


--- CALCUL DES PERFORMANCES ---
----------------------------------------
✅ Performance sur Domaine A (Kaggle) : 79.44%
✅ Performance sur Domaine B (Moi)    : 62.42%
----------------------------------------


In [ ]:
# =================================================================
# ÉTAPE 7 : ANALYSE ET VISUALISATIONS FINALES
# =================================================================

import matplotlib.pyplot as plt
import numpy as np

# --- 1. PRÉPARATION DES DONNÉES ---
iterations = [d["Iteration"] for d in history_data]
# Conversion des pourcentages (string) en nombres (float)
acc_A = [float(d["Acc Domaine A"].replace('%', '')) for d in history_data]
acc_B = [float(d["Acc Domaine B"].replace('%', '')) for d in history_data]

# --- 2. GRAPHIQUE 1 : COURBE DE PROGRESSION (ROBUSTESSE) ---
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1) # On crée deux graphiques côte à côte

plt.plot(iterations, acc_A, marker='o', ls='--', label='Domaine A (Kaggle)', color='#1f77b4', alpha=0.7)
plt.plot(iterations, acc_B, marker='s', ls='-', label='Domaine B (Tes mains)', color='#ff7f0e', lw=3)

# Annotation des scores sur la courbe B
for i, txt in enumerate(acc_B):
    plt.annotate(f"{txt}%", (iterations[i], acc_B[i]), textcoords="offset points", xytext=(0,10), ha='center')

plt.title("Évolution de l'Accuracy par itération", fontweight='bold')
plt.xlabel("Budget d'annotation")
plt.ylabel("Précision (%)")
plt.ylim(min(acc_B) - 10, 105)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend()

# --- 3. GRAPHIQUE 2 : COMPARAISON INITIAL VS FINAL (GAIN) ---
plt.subplot(1, 2, 2)

labels = ['Domaine A (Kaggle)', 'Domaine B (Moi)']
initial_scores = [acc_A[0], acc_B[0]]
final_scores = [acc_A[-1], acc_B[-1]]

x = np.arange(len(labels))
width = 0.35

plt.bar(x - width/2, initial_scores, width, label='Initial (Baseline)', color='#aec7e8')
plt.bar(x + width/2, final_scores, width, label='Final (Après AL)', color='#ffbb78')

plt.ylabel('Précision (%)')
plt.title('Gain de performance global', fontweight='bold')
plt.xticks(x, labels)
plt.legend()
plt.ylim(0, 110)

# Ajout du texte de progression
gain_B = acc_B[-1] - acc_B[0]
plt.text(1, acc_B[-1] + 2, f"Gain: +{gain_B:.1f}%", ha='center', color='darkorange', fontweight='bold')

# --- 4. SAUVEGARDE ET AFFICHAGE ---
plt.tight_layout()
plt.savefig("bilan_active_learning.png", dpi=300)
plt.show()

print(f"✅ Analyse terminée.")
print(f"📊 Gain net sur le Domaine B : {gain_B:.2f}%")

In [ ]:
# =================================================================
# ÉTAPE 7 (SUITE) : ÉTUDE COMPARATIVE (RANDOM vs UNCERTAINTY)
# =================================================================

# --- CONFIGURATION ---
BUDGET_RATIOS = [0.01, 0.05, 0.10, 0.20] # 1%, 5%, 10%, 20%
SEEDS = [0] 
BATCH_SIZE = 32
EPOCHS_FINE = 2 

# Stockage des moyennes pour le graphique
final_rnd = []
final_unc = []

print("🚀 COMPARAISON DES STRATÉGIES EN COURS...")

for ratio in BUDGET_RATIOS:
    k = int(ratio * len(X_pool_B))
    print(f"\n📊 Test avec Budget {int(ratio*100)}% ({k} images du Domaine B)")
    
    acc_rnd_list = []
    acc_unc_list = []
    
    for seed in SEEDS:
        # Configuration des graines pour la reproductibilité
        np.random.seed(seed)
        torch.manual_seed(seed)
        
        # --- 1. STRATÉGIE ALÉATOIRE ---
        idx_rnd = np.random.choice(len(X_pool_B), k, replace=False)
        ds_rnd = to_tensor(np.concatenate([X_train_A, X_pool_B[idx_rnd]]), 
                           np.concatenate([y_train_A, y_pool_B[idx_rnd]]))
        loader_rnd = DataLoader(ds_rnd, batch_size=BATCH_SIZE, shuffle=True)
        
        m_rnd = get_resnet_model().to(device)
        opt_rnd = torch.optim.AdamW(m_rnd.parameters(), lr=1e-4)
        m_rnd = train_model(m_rnd, loader_rnd, criterion, opt_rnd, epochs=EPOCHS_FINE, device=device)
        acc_rnd_list.append(get_accuracy(m_rnd, test_B_loader))
        
        # --- 2. STRATÉGIE INCERTITUDE (MARGIN) ---
        # On utilise le 'model' baseline pour la sélection
        idx_unc = margin_confidence(model, pool_B_ds, n=k)
        ds_unc = to_tensor(np.concatenate([X_train_A, X_pool_B[idx_unc]]), 
                           np.concatenate([y_train_A, y_pool_B[idx_unc]]))
        loader_unc = DataLoader(ds_unc, batch_size=BATCH_SIZE, shuffle=True)
        
        m_unc = get_resnet_model().to(device)
        opt_unc = torch.optim.AdamW(m_unc.parameters(), lr=1e-4)
        m_unc = train_model(m_unc, loader_unc, criterion, opt_unc, epochs=EPOCHS_FINE, device=device)
        acc_unc_list.append(get_accuracy(m_unc, test_B_loader))

    final_rnd.append(np.mean(acc_rnd_list))
    final_unc.append(np.mean(acc_unc_list))
    print(f"✅ Résultat : Random {final_rnd[-1]:.2f}% vs Uncertainty {final_unc[-1]:.2f}%")

# --- VISUALISATION COMPARATIVE ---
plt.figure(figsize=(10, 6))
labels_pct = [f"{int(r*100)}%" for r in BUDGET_RATIOS]

plt.plot(labels_pct, final_rnd, marker='o', label="Sélection Aléatoire", color='gray', linestyle='--')
plt.plot(labels_pct, final_unc, marker='s', label="Apprentissage Actif (Margin)", color='red', linewidth=2)



plt.title("Comparaison : Pourquoi utiliser l'Active Learning ?", fontsize=14)
plt.xlabel("Budget d'annotation (% du pool B)")
plt.ylabel("Accuracy sur Domaine B (%)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()